In [0]:
dbutils.widgets.text("catalog", "weather_dev", "Catálogo de destino")
catalog = dbutils.widgets.get("catalog")

In [0]:
# ============================================================
# 04_ml_temperature_model
# Modelo súper simple de regresión: predecir temperatura según
# la hora del día, usando MLflow para trackear y comparar 2
# modelos distintos, y elegir el mejor automáticamente.
# ============================================================

import mlflow
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


# ------------------------------------------------------------
# 1. Cargar y preparar los datos
# ------------------------------------------------------------

# Traemos la tabla silver (168 filas, una por hora de pronóstico)
# y la pasamos a pandas. Como es un dataset chiquito, no hace
# falta quedarnos trabajando en Spark para entrenar el modelo.
df = spark.table(f"{catalog}.silver.weather_forecast_hourly").toPandas()

# Feature engineering súper simple: extraemos la hora del día (0-23)
# a partir del timestamp del pronóstico. Va a ser nuestra única
# variable predictora, a propósito, para mantener esto mínimo.
df["hour_of_day"] = pd.to_datetime(df["forecast_time"]).dt.hour

# X = variable(s) que usamos para predecir (features)
# y = lo que queremos predecir (target): la temperatura en °C
X = df[["hour_of_day"]]
y = df["temperature_c"]

# Dividimos los datos: 80% para entrenar, 20% para evaluar con
# datos que el modelo nunca vio (evita hacer trampa al medir qué
# tan bueno es). random_state=42 solo asegura que la división sea
# reproducible cada vez que corras esta celda.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Le decimos a MLflow que registre automáticamente parámetros,
# métricas y el modelo entrenado de scikit-learn, sin tener que
# hacerlo todo a mano en cada modelo.
mlflow.sklearn.autolog()


# ------------------------------------------------------------
# 2. Modelo 1: Regresión lineal
# ------------------------------------------------------------

# Abrimos una "corrida" (run) de MLflow. Todo lo que pase dentro
# de este bloque queda asociado a esta corrida específica,
# versionada y consultable después en la UI de Experiments.
with mlflow.start_run(run_name="temperature_vs_hour_linear_regression"):

    # Regresión lineal clásica: busca la mejor línea recta que
    # relacione hora del día -> temperatura.
    model_linear = LinearRegression()
    model_linear.fit(X_train, y_train)

    # Predicciones sobre el set de prueba (datos no vistos)
    preds_linear = model_linear.predict(X_test)

    # Métricas de evaluación:
    # - MAE (Mean Absolute Error): en promedio, cuántos grados se
    #   equivoca el modelo (más bajo = mejor)
    # - R² (coeficiente de determinación): qué tan bien explica el
    #   modelo la variación de la temperatura (1.0 = perfecto, 0 = nulo)
    mae_linear = mean_absolute_error(y_test, preds_linear)
    r2_linear = r2_score(y_test, preds_linear)

    # Logueamos las métricas explícitamente con nombres consistentes
    # (test_mae, test_r2) para poder comparar contra el modelo 2 después.
    mlflow.log_metric("test_mae", mae_linear)
    mlflow.log_metric("test_r2", r2_linear)

    print("Modelo 1 — Regresión lineal")
    print(f"  MAE: {mae_linear:.2f} °C")
    print(f"  R²: {r2_linear:.2f}")
    print(f"  Coeficiente (hour_of_day): {model_linear.coef_[0]:.3f}")
    print(f"  Intercepto: {model_linear.intercept_:.3f}")


# ------------------------------------------------------------
# 3. Modelo 2: Árbol de decisión
# ------------------------------------------------------------

# Segundo modelo, distinto al primero, para tener con qué comparar.
# max_depth=3 lo mantiene chiquito y evita que memorice el set de
# entrenamiento (sobreajuste), dado que tenemos pocos datos.
with mlflow.start_run(run_name="temperature_vs_hour_decision_tree"):

    model_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
    model_tree.fit(X_train, y_train)

    preds_tree = model_tree.predict(X_test)
    mae_tree = mean_absolute_error(y_test, preds_tree)
    r2_tree = r2_score(y_test, preds_tree)

    # Mismos nombres de métrica que en el modelo 1, para que la
    # comparación de la siguiente celda funcione parejo.
    mlflow.log_metric("test_mae", mae_tree)
    mlflow.log_metric("test_r2", r2_tree)

    print("Modelo 2 — Árbol de decisión")
    print(f"  MAE: {mae_tree:.2f} °C")
    print(f"  R²: {r2_tree:.2f}")


# ------------------------------------------------------------
# 4. Comparar las corridas y elegir el mejor modelo
# ------------------------------------------------------------

# mlflow.search_runs() trae todas las corridas registradas en el
# experimento asociado a este notebook. Las ordenamos por test_mae
# ascendente, así que la primera fila es automáticamente la mejor
# (menor error = mejor modelo).
runs = mlflow.search_runs(order_by=["metrics.test_mae ASC"])

# Nos quedamos solo con las columnas relevantes para leer la
# comparación fácilmente.
comparison = runs[
    ["tags.mlflow.runName", "metrics.test_mae", "metrics.test_r2", "run_id"]
]
display(comparison)

# La primera fila, al estar ordenado ascendente por MAE, es el ganador.
best_run = runs.iloc[0]

print("Mejor modelo:", best_run["tags.mlflow.runName"])
print(f"   test_mae: {best_run['metrics.test_mae']:.3f}")
print(f"   test_r2: {best_run['metrics.test_r2']:.3f}")
print(f"   run_id: {best_run['run_id']}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.models")

In [0]:
from mlflow import MlflowClient

# Le decimos a MLflow que registre modelos en Unity Catalog
# (en vez del registro clásico de MLflow, que ya está deprecado
# a favor de UC — así el modelo queda gobernado igual que tus tablas)
mlflow.set_registry_uri("databricks-uc")

# Nombre del modelo con el namespace de 3 niveles: catálogo.schema.modelo
registered_model_name = f"{catalog}.models.temperature_predictor"

# El URI apunta al modelo guardado dentro de la corrida ganadora
model_uri = f"runs:/{best_run['run_id']}/model"

# Registramos una nueva versión de este modelo en Unity Catalog
model_version = mlflow.register_model(model_uri=model_uri, name=registered_model_name)

# Unity Catalog ya no usa "Stages" (Production/Staging) como el
# MLflow Registry clásico -- ahora se usan alias. Marcamos esta
# versión con el alias "champion", que es la forma moderna de decir
# "este es el modelo que está en producción / el que se debe usar".
client = MlflowClient()
client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=model_version.version
)

print(f"✅ Modelo registrado: {registered_model_name}, versión {model_version.version}")
print(f"   Alias 'champion' asignado — este es el modelo 'en producción'")